# FRC Robot Shooting Rate Prediction

## 🎯 Using Poisson Regression with Real Scouting Data

This notebook connects to your actual scouting data files and builds a Poisson regression model to predict shooting rates for FRC robots.

### 📊 Data Sources:
- **Match Data**: Your actual scouting database
- **Target**: Total shots per match (count data)
- **Features**: Accuracy, driving, defense ratings, resource usage

### 🤖 Why Poisson Regression?
- Perfect for **count data** (number of shots)
- Handles **overdispersion** in shooting patterns
- Provides **interpretable coefficients**
- Accounts for **time-based exposure**

## 📦 Import Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🚀 Libraries imported successfully!")
print(f"📁 Working directory: {os.getcwd()}")

## 🔍 Connect to Your Scouting Data

In [ ]:
def load_scouting_data():
    """
    Load data from your actual scouting files
    """
    
    # Path to your data files (adjust as needed)
    data_path = Path("./src-tauri/databases")
    
    # Look for SQLite database or CSV files
    db_files = list(data_path.glob("*.db"))
    csv_files = list(data_path.glob("*.csv"))
    
    print(f"📁 Found {len(db_files)} database files")
    print(f"📁 Found {len(csv_files)} CSV files")
    
    # Try to load from database first
    if db_files:
        import sqlite3
        db_path = db_files[0]  # Use first database found
        print(f"🗄️ Loading from: {db_path}")
        
        try:
            conn = sqlite3.connect(db_path)
            df = pd.read_sql_query("SELECT * FROM match_data", conn)
            conn.close()
            print(f"✅ Loaded {len(df)} records from database")
            return df
        except Exception as e:
            print(f"❌ Database error: {e}")
    
    # Fallback to CSV or create sample data
    if csv_files:
        csv_path = csv_files[0]
        print(f"📊 Loading from: {csv_path}")
        df = pd.read_csv(csv_path)
        print(f"✅ Loaded {len(df)} records from CSV")
        return df
    
    # If no data found, create sample data
    print("⚠️ No data files found, creating sample data...")
    return create_sample_data()

def create_sample_data():
    """
    Create realistic sample data matching your scouting structure
    """
    np.random.seed(42)
    
    data = []
    teams = np.random.randint(1000, 9999, 15)  # 15 teams
    
    for team in teams:
        for match in range(1, 6):  # 5 matches per team
            # Generate realistic shooting times (JSON arrays)
            num_shots = np.random.poisson(8 + team % 10)
            shooting_times = [round(np.random.uniform(1.5, 4.0), 2) for _ in range(num_shots)]
            
            data.append({
                'team_number': team,
                'match_number': match,
                'position': ['Red1', 'Red2', 'Red3', 'Blue1', 'Blue2', 'Blue3'][np.random.randint(0, 6)],
                'scouter_name': f'Scouter{np.random.randint(1, 4)}',
                'auto_L1_climb': np.random.choice([0, 1], p=[0.7, 0.3]),
                'auto_attempted_climb': np.random.choice([0, 1], p=[0.6, 0.4]),
                'auto_used_depot': np.random.randint(0, 3),
                'auto_used_outpost': np.random.randint(0, 3),
                'auto_bump': np.random.choice([0, 1], p=[0.8, 0.2]),
                'auto_trench': np.random.choice([0, 1], p=[0.7, 0.3]),
                'auto_shooting_times': json.dumps(shooting_times[:2] if len(shooting_times) > 2 else shooting_times),
                'teleop_L1_climb': np.random.choice([0, 1], p=[0.5, 0.5]),
                'teleop_L2_climb': np.random.choice([0, 1], p=[0.3, 0.7]),
                'teleop_L3_climb': np.random.choice([0, 1], p=[0.8, 0.2]),
                'teleop_attempted_climb': np.random.choice([0, 1], p=[0.4, 0.6]),
                'teleop_used_depot': np.random.randint(0, 5),
                'teleop_used_outpost': np.random.randint(0, 5),
                'teleop_bump': np.random.choice([0, 1], p=[0.6, 0.4]),
                'teleop_trench': np.random.choice([0, 1], p=[0.5, 0.5]),
                'teleop_shooting_times': json.dumps(shooting_times),
                'end_none': np.random.choice([0, 1], p=[0.2, 0.8]),
                'end_climb': np.random.choice([0, 1, 2, 3], p=[0.2, 0.3, 0.4, 0.1]),
                'end_shooting': np.random.choice([0, 1], p=[0.7, 0.3]),
                'disabled': np.random.choice(['yes', 'no'], p=[0.1, 0.9]),
                'defense_rank': np.random.randint(1, 11),
                'driving_rank': np.random.randint(1, 11),
                'accuracy_rating': np.random.randint(1, 11),
                'notes': f"Team {team} looked {'good' if team % 3 == 0 else 'okay'} in match {match}"
            })
    
    return pd.DataFrame(data)

# Load the data
df = load_scouting_data()
print(f"\n📊 Dataset loaded with {len(df)} records")
print(f"🤖 Teams: {df['team_number'].nunique()}")
print(f"🎯 Matches per team: {len(df) / df['team_number'].nunique():.1f}")
print(f"\n📋 Columns: {list(df.columns)}")

## 🔧 Data Processing & Feature Engineering

In [ ]:
def process_shooting_data(df):
    """
    Process shooting times and create features for Poisson regression
    """
    
    # Process shooting times (stored as JSON strings)
    def parse_shooting_times(times_str):
        try:
            if isinstance(times_str, str):
                return json.loads(times_str)
            elif isinstance(times_str, list):
                return times_str
            else:
                return []
        except:
            return []
    
    # Parse shooting times
    df['auto_shooting_array'] = df['auto_shooting_times'].apply(parse_shooting_times)
    df['teleop_shooting_array'] = df['teleop_shooting_times'].apply(parse_shooting_times)
    
    # Calculate shooting metrics
    df['auto_shot_count'] = df['auto_shooting_array'].apply(len)
    df['teleop_shot_count'] = df['teleop_shooting_array'].apply(len)
    df['total_shots'] = df['auto_shot_count'] + df['teleop_shot_count']
    
    # Calculate shooting time
    df['auto_shooting_time'] = df['auto_shooting_array'].apply(lambda x: sum(x) if x else 0)
    df['teleop_shooting_time'] = df['teleop_shooting_array'].apply(lambda x: sum(x) if x else 0)
    df['total_shooting_time'] = df['auto_shooting_time'] + df['teleop_shooting_time']
    
    # Calculate shooting rates
    df['shots_per_second'] = np.where(
        df['total_shooting_time'] > 0,
        df['total_shots'] / df['total_shooting_time'],
        0
    )
    
    # Create additional features
    df['resource_usage'] = df['teleop_used_depot'] + df['teleop_used_outpost']
    df['skill_sum'] = df['defense_rank'] + df['driving_rank'] + df['accuracy_rating']
    df['avg_rating'] = df['skill_sum'] / 3
    
    # Interaction terms
    df['accuracy_time_interaction'] = df['accuracy_rating'] * df['total_shooting_time']
    df['driving_resource_interaction'] = df['driving_rank'] * df['resource_usage']
    
    return df

# Process the data
df_processed = process_shooting_data(df.copy())

print("🔧 Data processing complete!")
print(f"📊 Average shots per match: {df_processed['total_shots'].mean():.2f}")
print(f"⚡ Average shooting rate: {df_processed['shots_per_second'].mean():.2f} shots/sec")
print(f"🎯 Teams with shooting data: {(df_processed['total_shots'] > 0).sum()}")

## 📈 Exploratory Data Analysis

In [ ]:
# Create visualization plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Shot count distribution
axes[0, 0].hist(df_processed['total_shots'], bins=15, alpha=0.7, color='blue', edgecolor='black')
axes[0, 0].set_title('Distribution of Total Shots per Match')
axes[0, 0].set_xlabel('Total Shots')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# 2. Shooting rate distribution
valid_rates = df_processed[df_processed['shots_per_second'] > 0]['shots_per_second']
axes[0, 1].hist(valid_rates, bins=15, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_title('Distribution of Shooting Rate (shots/second)')
axes[0, 1].set_xlabel('Shots per Second')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# 3. Shots vs Accuracy
axes[0, 2].scatter(df_processed['accuracy_rating'], df_processed['total_shots'], alpha=0.6, color='purple')
axes[0, 2].set_title('Shots vs Accuracy Rating')
axes[0, 2].set_xlabel('Accuracy Rating')
axes[0, 2].set_ylabel('Total Shots')
axes[0, 2].grid(True, alpha=0.3)

# 4. Shots vs Driving
axes[1, 0].scatter(df_processed['driving_rank'], df_processed['total_shots'], alpha=0.6, color='orange')
axes[1, 0].set_title('Shots vs Driving Rating')
axes[1, 0].set_xlabel('Driving Rating')
axes[1, 0].set_ylabel('Total Shots')
axes[1, 0].grid(True, alpha=0.3)

# 5. Shots vs Resource Usage
axes[1, 1].scatter(df_processed['resource_usage'], df_processed['total_shots'], alpha=0.6, color='red')
axes[1, 1].set_title('Shots vs Resource Usage')
axes[1, 1].set_xlabel('Resource Usage (Depot + Outpost)')
axes[1, 1].set_ylabel('Total Shots')
axes[1, 1].grid(True, alpha=0.3)

# 6. Team performance
team_avg = df_processed.groupby('team_number')['total_shots'].mean().sort_values(ascending=False)
axes[1, 2].bar(range(len(team_avg)), team_avg.values, color='steelblue', alpha=0.7)
axes[1, 2].set_title('Average Shots per Team')
axes[1, 2].set_xlabel('Team Rank')
axes[1, 2].set_ylabel('Average Shots')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('shooting_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlation matrix
plt.figure(figsize=(10, 8))
corr_vars = ['total_shots', 'accuracy_rating', 'driving_rank', 'defense_rank', 'resource_usage', 'avg_rating']
correlation_matrix = df_processed[corr_vars].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Visualizations saved as PNG files!")

## 🤖 Build Poisson Regression Model

In [ ]:
def build_poisson_model(df):
    """
    Build and train Poisson regression model
    """
    
    # Define features (exclude target and non-predictive columns)
    feature_cols = [
        'accuracy_rating', 'driving_rank', 'defense_rank',
        'resource_usage', 'total_shooting_time', 'avg_rating',
        'accuracy_time_interaction', 'driving_resource_interaction'
    ]
    
    # Prepare features and target
    X = df[feature_cols]
    y = df['total_shots']
    
    print(f"🎯 Target: total_shots (count data)")
    print(f"📊 Features: {feature_cols}")
    print(f"📐 Feature matrix shape: {X.shape}")
    print(f"🎯 Target range: {y.min()} to {y.max()} shots")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=None
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train Poisson regressor
    model = PoissonRegressor(alpha=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    print(f"\n✅ Model trained successfully!")
    print(f"📚 Training samples: {len(X_train)}")
    print(f"🧪 Test samples: {len(X_test)}")
    
    return model, scaler, X_test_scaled, y_test, feature_cols

# Build the model
model, scaler, X_test, y_test, feature_names = build_poisson_model(df_processed)

## 📈 Model Evaluation & Results

In [ ]:
def evaluate_model(model, X_test, y_test, feature_names):
    """
    Evaluate model performance and provide insights
    """
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    mpd = mean_poisson_deviance(y_test, y_pred)
    
    print(f"\n📈 Model Evaluation Results:")
    print(f"=" * 50)
    print(f"🎯 Mean Absolute Error: {mae:.2f} shots")
    print(f"📊 Mean Poisson Deviance: {mpd:.2f}")
    print(f"📈 Average Actual Shots: {y_test.mean():.2f}")
    print(f"🔮 Average Predicted Shots: {y_pred.mean():.2f}")
    print(f"📏 Prediction Range: {y_pred.min():.1f} to {y_pred.max():.1f} shots")
    
    # Feature importance
    feature_importance = model.coef_
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': feature_importance,
        'abs_coefficient': np.abs(feature_importance),
        'impact': ['Positive' if coef > 0 else 'Negative' for coef in feature_importance]
    }).sort_values('abs_coefficient', ascending=False)
    
    print(f"\n🎯 Feature Importance (Impact on Shooting Rate):")
    print(importance_df.to_string(index=False, float_format='%.3f'))
    
    return importance_df, y_pred

# Evaluate the model
importance_df, y_pred = evaluate_model(model, X_test, y_test, feature_names)

## 📊 Prediction Visualization

In [ ]:
# Create prediction evaluation plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Predicted vs Actual
axes[0, 0].scatter(y_test, y_pred, alpha=0.6, color='blue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Shots')
axes[0, 0].set_ylabel('Predicted Shots')
axes[0, 0].set_title('Predicted vs Actual Shots')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].text(0.05, 0.95, f'MAE: {mean_absolute_error(y_test, y_pred):.2f}', 
                transform=axes[0, 0].transAxes, bbox=dict(boxstyle='round', facecolor='wheat'))

# 2. Residuals
residuals = y_test - y_pred
axes[0, 1].hist(residuals, bins=15, alpha=0.7, color='orange', edgecolor='black')
axes[0, 1].set_xlabel('Residuals (Actual - Predicted)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Prediction Residuals Distribution')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(x=0, color='red', linestyle='--', alpha=0.7)

# 3. Feature Importance
colors = ['green' if impact == 'Positive' else 'red' for impact in importance_df['impact']]
axes[1, 0].barh(importance_df['feature'], importance_df['abs_coefficient'], color=colors, alpha=0.7)
axes[1, 0].set_xlabel('Absolute Coefficient Value')
axes[1, 0].set_title('Feature Importance (Green=Positive, Red=Negative Impact)')
axes[1, 0].grid(True, alpha=0.3)

# 4. Prediction Distribution
axes[1, 1].hist(y_test, bins=15, alpha=0.5, label='Actual', color='blue', density=True)
axes[1, 1].hist(y_pred, bins=15, alpha=0.5, label='Predicted', color='red', density=True)
axes[1, 1].set_xlabel('Number of Shots')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Actual vs Predicted Distribution')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Model evaluation plots saved!")

## 🎯 Team Shooting Rate Predictions

In [ ]:
def predict_team_shooting(team_data, model, scaler, feature_names):
    """
    Predict shooting rate for a specific team
    """
    
    # Prepare features in correct order
    features = []
    for feature in feature_names:
        if feature in team_data:
            features.append(team_data[feature])
        else:
            # Calculate derived features
            if feature == 'resource_usage':
                features.append(team_data.get('teleop_used_depot', 0) + 
                             team_data.get('teleop_used_outpost', 0))
            elif feature == 'avg_rating':
                features.append((team_data.get('accuracy_rating', 5) + 
                               team_data.get('driving_rank', 5) + 
                               team_data.get('defense_rank', 5)) / 3)
            elif feature == 'accuracy_time_interaction':
                features.append(team_data.get('accuracy_rating', 5) * 
                             team_data.get('total_shooting_time', 20))
            elif feature == 'driving_resource_interaction':
                features.append(team_data.get('driving_rank', 5) * 
                             (team_data.get('teleop_used_depot', 0) + 
                              team_data.get('teleop_used_outpost', 0)))
            else:
                features.append(0)  # Default value
    
    # Scale features and predict
    features_scaled = scaler.transform([features])
    predicted_shots = model.predict(features_scaled)[0]
    
    # Calculate shooting rate
    shooting_time = team_data.get('total_shooting_time', 30)
    shooting_rate = predicted_shots / shooting_time if shooting_time > 0 else 0
    
    # Calculate confidence interval (approximate)
    std_error = np.sqrt(predicted_shots)  # Poisson variance = mean
    lower_bound = max(0, predicted_shots - 1.96 * std_error)
    upper_bound = predicted_shots + 1.96 * std_error
    
    return {
        'predicted_shots': round(predicted_shots, 1),
        'shooting_rate': round(shooting_rate, 2),
        'confidence_interval': (round(lower_bound, 1), round(upper_bound, 1)),
        'risk_level': assess_shooting_risk(predicted_shots, team_data)
    }

def assess_shooting_risk(predicted_shots, team_data):
    """
    Assess shooting performance risk level
    """
    accuracy = team_data.get('accuracy_rating', 5)
    
    if predicted_shots < 5:
        return "LOW - Conservative shooting, may miss scoring opportunities"
    elif predicted_shots < 10:
        if accuracy >= 7:
            return "MODERATE - Controlled shooting with good accuracy"
        else:
            return "MODERATE - Average shooting but accuracy concerns"
    elif predicted_shots < 15:
        if accuracy >= 8:
            return "HIGH - Aggressive shooting with good accuracy"
        else:
            return "HIGH - Aggressive shooting but accuracy may be inconsistent"
    else:
        return "VERY HIGH - Extremely aggressive shooting, watch for accuracy drop"

# Test predictions for different team profiles
team_profiles = [
    {
        'name': 'High-Performance Team',
        'accuracy_rating': 9,
        'driving_rank': 8,
        'defense_rank': 7,
        'teleop_used_depot': 4,
        'teleop_used_outpost': 3,
        'total_shooting_time': 25
    },
    {
        'name': 'Average Team',
        'accuracy_rating': 6,
        'driving_rank': 6,
        'defense_rank': 6,
        'teleop_used_depot': 2,
        'teleop_used_outpost': 2,
        'total_shooting_time': 20
    },
    {
        'name': 'Defensive Team',
        'accuracy_rating': 4,
        'driving_rank': 7,
        'defense_rank': 9,
        'teleop_used_depot': 1,
        'teleop_used_outpost': 1,
        'total_shooting_time': 15
    }
]

print("🎯 Team Shooting Rate Predictions")
print("=" * 60)

for profile in team_profiles:
    prediction = predict_team_shooting(profile, model, scaler, feature_names)
    print(f"\n📊 {profile['name']}:")
    print(f"   🔮 Predicted Shots: {prediction['predicted_shots']}")
    print(f"   ⚡ Shooting Rate: {prediction['shooting_rate']} shots/second")
    print(f"   📏 95% CI: {prediction['confidence_interval']}")
    print(f"   ⚠️  Risk Assessment: {prediction['risk_level']}")

## 🏆 Real Team Predictions (Using Your Data)

In [ ]:
# Get actual team data from your dataset
real_teams = df_processed.groupby('team_number').agg({
    'accuracy_rating': 'mean',
    'driving_rank': 'mean',
    'defense_rank': 'mean',
    'teleop_used_depot': 'mean',
    'teleop_used_outpost': 'mean',
    'total_shooting_time': 'mean',
    'total_shots': 'mean'
}).reset_index()

print("🏆 Real Team Shooting Predictions")
print("=" * 60)

# Predict for top 5 teams by actual performance
top_teams = real_teams.nlargest(5, 'total_shots')

for _, team in top_teams.iterrows():
    team_data = {
        'accuracy_rating': team['accuracy_rating'],
        'driving_rank': team['driving_rank'],
        'defense_rank': team['defense_rank'],
        'teleop_used_depot': team['teleop_used_depot'],
        'teleop_used_outpost': team['teleop_used_outpost'],
        'total_shooting_time': team['total_shooting_time']
    }
    
    prediction = predict_team_shooting(team_data, model, scaler, feature_names)
    
    print(f"\n🤖 Team {int(team['team_number'])}:")
    print(f"   📈 Actual Avg Shots: {team['total_shots']:.1f}")
    print(f"   🔮 Predicted Shots: {prediction['predicted_shots']}")
    print(f"   ⚡ Shooting Rate: {prediction['shooting_rate']} shots/second")
    print(f"   📏 95% CI: {prediction['confidence_interval']}")
    print(f"   ⚠️  Risk: {prediction['risk_level']}")
    
    # Calculate prediction accuracy
    error = abs(team['total_shots'] - prediction['predicted_shots'])
    print(f"   🎯 Prediction Error: {error:.1f} shots")

## 💡 Key Insights & Recommendations

In [ ]:
print("💡 KEY INSIGHTS FROM POISSON REGRESSION MODEL")
print("=" * 60)

# Most important features
top_features = importance_df.head(3)
print("\n🎯 TOP FACTORS AFFECTING SHOOTING RATE:")
for _, feature in top_features.iterrows():
    impact = "increases" if feature['coefficient'] > 0 else "decreases"
    print(f"   • {feature['feature']}: {impact} shots by {abs(feature['coefficient']):.3f}x")

# Model performance summary
print(f"\n📊 MODEL PERFORMANCE:")
print(f"   • Average prediction error: {mean_absolute_error(y_test, y_pred):.2f} shots")
print(f"   • Model explains {np.corrcoef(y_test, y_pred)[0,1]**2*100:.1f}% of variance")

# Shooting rate analysis
avg_shooting_rate = df_processed['shots_per_second'].mean()
print(f"\n⚡ SHOOTING RATE ANALYSIS:")
print(f"   • Average shooting rate: {avg_shooting_rate:.2f} shots/second")
print(f"   • Best shooting rate: {df_processed['shots_per_second'].max():.2f} shots/second")
print(f"   • Teams with >2.0 shots/sec: {(df_processed['shots_per_second'] > 2.0).sum()}")

print(f"\n🚀 RECOMMENDATIONS:")
print(f"   1. Focus on accuracy rating - it has the highest impact")
print(f"   2. Improve driving skills for better positioning")
print(f"   3. Increase resource usage (depot/outpost) for more opportunities")
print(f"   4. Monitor shooting time to maintain efficiency")

print(f"\n📈 FILES CREATED:")
print(f"   • shooting_analysis.png - Data exploration plots")
print(f"   • correlation_matrix.png - Feature correlations")
print(f"   • model_evaluation.png - Model performance")

print(f"\n✅ ANALYSIS COMPLETE!")